In [0]:
# ── Cube views are virtual slices of Gold KPI tables ─────────────────────────
# No data stored — computed on query
# Think of each view as one BI dashboard panel

# ── CUBE 1 — Premium vs Free genre preference ─────────────────────────────────
spark.sql("""
    CREATE OR REPLACE VIEW spotify_catalog.view.cube_premium_vs_free_genre AS
    SELECT
        genre,
        subscription_type,
        total_streams,
        unique_users,
        meaningful_play_rate_pct,
        RANK() OVER (
            PARTITION BY subscription_type
            ORDER BY total_streams DESC
        ) AS genre_rank_in_plan
    FROM spotify_catalog.gold.genre_subscription_kpi
    ORDER BY subscription_type, genre_rank_in_plan
""")

# ── CUBE 2 — Top 10 tracks overall ───────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE VIEW spotify_catalog.view.cube_top10_tracks AS
    SELECT
        track_name,
        artist_name,
        genre,
        total_streams,
        unique_listeners,
        total_listen_hrs,
        avg_completion_pct,
        skip_rate_pct,
        RANK() OVER (ORDER BY total_listen_hrs DESC) AS track_rank
    FROM spotify_catalog.gold.top_tracks_kpi
    QUALIFY RANK() OVER (ORDER BY total_listen_hrs DESC) <= 10
""")

# ── CUBE 3 — Monthly growth trend ─────────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE VIEW spotify_catalog.view.cube_monthly_growth AS
    SELECT
        year,
        month,
        total_streams,
        active_users,
        total_listen_hrs,
        streams_per_user,
        -- month over month growth
        LAG(total_streams) OVER (ORDER BY year, month)  AS prev_month_streams,
        ROUND(
            (total_streams - LAG(total_streams) OVER (ORDER BY year, month))
            / LAG(total_streams) OVER (ORDER BY year, month) * 100
        , 2)                                            AS mom_growth_pct
    FROM spotify_catalog.gold.monthly_trend_kpi
    ORDER BY year, month
""")

# ── CUBE 4 — Device preference by plan ────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE VIEW spotify_catalog.view.cube_device_preference AS
    SELECT
        subscription_type,
        device_type,
        total_streams,
        unique_users,
        avg_listen_sec,
        abandon_rate_pct,
        RANK() OVER (
            PARTITION BY subscription_type
            ORDER BY total_streams DESC
        ) AS device_rank_in_plan
    FROM spotify_catalog.gold.device_subscription_kpi
    ORDER BY subscription_type, device_rank_in_plan
""")

# ── CUBE 5 — Platinum users leaderboard ───────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE VIEW spotify_catalog.view.cube_platinum_users AS
    SELECT
        user_name,
        user_country,
        subscription_type,
        total_streams,
        meaningful_plays,
        engagement_score,
        rank_in_tier
    FROM spotify_catalog.gold.user_engagement_kpi
     where rank_in_tier <=10
    ORDER BY engagement_score DESC
""")

print("All Gold cube views created")